# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [ ]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
import time
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

# Work whether the kernel starts in the project root or notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(PROJECT_ROOT / ".env")

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print(f"📂 Project root: {PROJECT_ROOT}")

In [ ]:
# OpenAI Setup for Vocareum
import openai

openai_api_key = os.getenv("OPENAI_API_KEY")
client = None
if not openai_api_key:
    print("⚠️ No OPENAI_API_KEY found; data processing remains available, but AI stages are disabled.")
else:
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key,
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")

In [ ]:
# Import implemented foundation components and agents
from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data,
)
from risk_analyst_agent import RiskAnalystAgent
from compliance_officer_agent import ComplianceOfficerAgent

explainability_logger = ExplainabilityLogger(
    str(OUTPUT_DIR / "audit_logs" / "workflow_integration.jsonl")
)
risk_agent = RiskAnalystAgent(client, explainability_logger) if client else None
compliance_agent = ComplianceOfficerAgent(client, explainability_logger) if client else None

print("✅ Workflow components imported")
print(f"🤖 AI agents: {'ready' if client else 'disabled until an API key is configured'}")

## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [ ]:
def load_and_preprocess_data(data_dir=DATA_DIR):
    """Load CSV inputs, normalize optional values, and return record dictionaries."""
    data_dir = Path(data_dir)
    required_files = {
        "customers": data_dir / "customers.csv",
        "accounts": data_dir / "accounts.csv",
        "transactions": data_dir / "transactions.csv",
    }
    missing = [str(path) for path in required_files.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing required data files: {', '.join(missing)}")

    customers_df = pd.read_csv(required_files["customers"], dtype={"ssn_last_4": str})
    accounts_df = pd.read_csv(required_files["accounts"])
    transactions_df = pd.read_csv(required_files["transactions"])

    # Empty strings are easier to inspect in the notebook and are accepted by the schemas.
    for column in ("phone", "occupation"):
        if column in customers_df:
            customers_df[column] = customers_df[column].fillna("")
    for column in ("counterparty", "location"):
        if column in transactions_df:
            transactions_df[column] = transactions_df[column].fillna("")

    customers_data = customers_df.where(pd.notna(customers_df), None).to_dict("records")
    accounts_data = accounts_df.where(pd.notna(accounts_df), None).to_dict("records")
    transactions_data = transactions_df.where(pd.notna(transactions_df), None).to_dict("records")
    print(
        f"📈 Loaded {len(customers_data)} customers, {len(accounts_data)} accounts, "
        f"and {len(transactions_data)} transactions"
    )
    return customers_data, accounts_data, transactions_data


customers_data, accounts_data, transactions_data = load_and_preprocess_data()

## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [ ]:
def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=5):
    """Rank customers using transparent AML indicators and return the top candidates."""
    if top_n < 1:
        return []
    customers_data = customers_data or []
    accounts_data = accounts_data or []
    transactions_data = transactions_data or []

    accounts_by_customer = {}
    for account in accounts_data:
        accounts_by_customer.setdefault(account["customer_id"], []).append(account)
    transactions_by_account = {}
    for transaction in transactions_data:
        transactions_by_account.setdefault(transaction["account_id"], []).append(transaction)

    selected = []
    risk_points = {"Low": 0, "Medium": 2, "High": 4}
    now = pd.Timestamp.now(tz="UTC").tz_localize(None)
    for customer in customers_data:
        accounts = accounts_by_customer.get(customer["customer_id"], [])
        transactions = [
            transaction
            for account in accounts
            for transaction in transactions_by_account.get(account["account_id"], [])
        ]
        total_amount = sum(abs(float(txn["amount"])) for txn in transactions)
        transaction_count = len(transactions)
        dates = pd.to_datetime(
            [txn["transaction_date"] for txn in transactions], errors="coerce"
        )
        latest_activity = max((date for date in dates if not pd.isna(date)), default=None)
        recent_activity = latest_activity is not None and (now - latest_activity).days <= 90

        risk_flags = []
        if customer.get("risk_rating") in {"Medium", "High"}:
            risk_flags.append("elevated_risk_rating")
        if total_amount > 100_000:
            risk_flags.append("large_transaction_volume")
        if transaction_count > 50:
            risk_flags.append("high_transaction_frequency")
        if recent_activity:
            risk_flags.append("recent_activity")

        score = (
            risk_points.get(customer.get("risk_rating"), 0)
            + min(total_amount / 100_000, 4)
            + min(transaction_count / 50, 3)
            + int(recent_activity)
        )
        if len(risk_flags) >= 2:
            selected.append({
                "customer": customer,
                "accounts": accounts,
                "transactions": transactions,
                "total_amount": total_amount,
                "transaction_count": transaction_count,
                "latest_activity": latest_activity.isoformat() if latest_activity else None,
                "risk_flags": risk_flags,
                "risk_score": round(score, 3),
            })

    selected.sort(key=lambda item: (item["risk_score"], item["total_amount"]), reverse=True)
    print(f"📊 Selected {min(len(selected), top_n)} of {len(customers_data)} customers for analysis")
    return selected[:top_n]


selected_customers = screen_high_risk_customers(
    customers_data, accounts_data, transactions_data
)

## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [ ]:
def _normalize_review_decision(raw_decision, default_reviewer_id):
    """Normalize interactive strings or structured reviewer responses."""
    if isinstance(raw_decision, dict):
        response = str(raw_decision.get("decision", "")).strip().lower()
        reviewer_id = str(raw_decision.get("reviewer_id") or default_reviewer_id)
        rationale = str(raw_decision.get("rationale") or "")
    else:
        response = str(raw_decision).strip().lower()
        reviewer_id = default_reviewer_id
        rationale = ""
    approved_values = {"yes", "y", "approve", "approved", "proceed"}
    rejected_values = {"no", "n", "reject", "rejected", "decline"}
    if response not in approved_values | rejected_values:
        raise ValueError(f"Unsupported reviewer decision: {response!r}")
    return {
        "decision": "PROCEED" if response in approved_values else "REJECT",
        "raw_decision": response,
        "reviewer_id": reviewer_id,
        "rationale": rationale or "No additional reviewer rationale provided.",
        "decision_timestamp": datetime.now(timezone.utc).isoformat(),
    }


def run_two_stage_sar_workflow(
    selected_customers, decision_provider=None, reviewer_id="notebook_reviewer"
):
    """Run risk analysis, human review, and approved narrative generation.

    ``decision_provider`` receives ``(case_data, risk_analysis)`` and may return a yes/no
    string or a dictionary containing decision, reviewer_id, and rationale. Without a
    provider, the notebook prompts interactively.
    """
    if risk_agent is None or compliance_agent is None:
        raise RuntimeError("Configure OPENAI_API_KEY and rerun the setup cells before AI processing")

    processed_cases, approved_sars, rejected_cases, audit_decisions = [], [], [], []
    loader = DataLoader(explainability_logger)
    for index, candidate in enumerate(selected_customers or [], 1):
        started = time.perf_counter()
        case_data = None
        stage = "case_creation"
        try:
            case_data = loader.create_case_from_data(
                candidate["customer"], candidate["accounts"], candidate["transactions"]
            )
            processed_cases.append(case_data)
            stage = "risk_analysis"
            risk_analysis = risk_agent.analyze_case(case_data)
            print(f"\n🔍 {index}/{len(selected_customers)} — {case_data.customer.name}")
            print(
                f"Classification: {risk_analysis.classification} | "
                f"Risk: {risk_analysis.risk_level} | Confidence: {risk_analysis.confidence_score:.2f}"
            )

            stage = "human_review"
            raw_decision = (
                decision_provider(case_data, risk_analysis)
                if decision_provider
                else input("Proceed with SAR filing? (yes/no): ")
            )
            review = _normalize_review_decision(raw_decision, reviewer_id)
            should_proceed = review["decision"] == "PROCEED"
            record = {
                "case_id": case_data.case_id,
                "customer_name": case_data.customer.name,
                **review,
                "ai_classification": risk_analysis.classification,
                "ai_confidence": risk_analysis.confidence_score,
                "ai_risk_level": risk_analysis.risk_level,
                "processing_time_ms": 0.0,
                "error": None,
            }

            review_elapsed_ms = (time.perf_counter() - started) * 1000
            explainability_logger.log_agent_action(
                agent_type="HumanReviewer",
                action="sar_filing_decision",
                case_id=case_data.case_id,
                input_data={"risk_analysis": risk_analysis.model_dump()},
                output_data={
                    "decision": review["decision"],
                    "reviewer_id": review["reviewer_id"],
                    "rationale": review["rationale"],
                    "decision_timestamp": review["decision_timestamp"],
                },
                reasoning=review["rationale"],
                execution_time_ms=review_elapsed_ms,
            )

            if should_proceed:
                stage = "compliance_generation"
                compliance_review = compliance_agent.generate_compliance_narrative(
                    case_data, risk_analysis
                )
                stage = "sar_filing"
                sar_document = create_sar_document(
                    case_data, risk_analysis, compliance_review, human_review=review
                )
                record["sar_path"] = str(save_sar_document(sar_document))
                record["sar_id"] = sar_document["sar_metadata"]["sar_id"]
                approved_sars.append(sar_document)
                print(f"✅ SAR FILED: {record['sar_id']}")
            else:
                rejection = {
                    "case_id": case_data.case_id,
                    "customer_name": case_data.customer.name,
                    **review,
                    "reason": "human_rejection",
                }
                rejected_cases.append(rejection)
                print("❌ Case rejected by human reviewer; compliance generation skipped")

            record["processing_time_ms"] = (time.perf_counter() - started) * 1000
            audit_decisions.append(record)
        except Exception as exc:
            case_id = getattr(case_data, "case_id", "uncreated")
            record = {
                "case_id": case_id,
                "customer_name": candidate.get("customer", {}).get("name", "unknown"),
                "decision": "ERROR",
                "failed_stage": stage,
                "processing_time_ms": (time.perf_counter() - started) * 1000,
                "error": str(exc),
            }
            audit_decisions.append(record)
            explainability_logger.log_agent_action(
                agent_type="WorkflowOrchestrator",
                action="process_case",
                case_id=case_id,
                input_data={"customer_name": record["customer_name"], "stage": stage},
                output_data={"decision": "ERROR"},
                reasoning=f"Workflow failed during {stage}",
                execution_time_ms=record["processing_time_ms"],
                success=False,
                error_message=str(exc),
            )
            print(f"❌ Error processing {record['customer_name']} during {stage}: {exc}")

    return processed_cases, approved_sars, rejected_cases, audit_decisions


# Deliberately do not invoke the live, interactive workflow during Run All.
processed_cases, approved_sars, rejected_cases, audit_decisions = [], [], [], []
print("✅ Workflow ready. Call run_two_stage_sar_workflow(selected_customers) to begin review.")

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [ ]:
def create_sar_document(case_data, risk_analysis, compliance_review, human_review=None):
    """Create a serializable SAR package with decision and integrity metadata."""
    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now(timezone.utc).isoformat()
    human_review = human_review or {
        "decision": "PROCEED",
        "reviewer_id": "unspecified_reviewer",
        "rationale": "Human approval confirmed.",
        "decision_timestamp": filing_date,
    }
    document = {
        "sar_metadata": {
            "sar_id": sar_id,
            "case_id": case_data.case_id,
            "filing_date": filing_date,
            "filing_type": "Suspicious Activity Report",
            "ai_generated": True,
            "review_status": "human_approved",
            "schema_version": "1.1",
        },
        "subject_information": {
            "customer_name": case_data.customer.name,
            "customer_id": case_data.customer.customer_id,
            "address": case_data.customer.address,
            "date_of_birth": case_data.customer.date_of_birth,
            "ssn_last_4": case_data.customer.ssn_last_4,
            "customer_since": case_data.customer.customer_since,
            "risk_rating": case_data.customer.risk_rating,
            "account_ids": [account.account_id for account in case_data.accounts],
        },
        "suspicious_activity": {
            "classification": risk_analysis.classification,
            "risk_level": risk_analysis.risk_level,
            "confidence_score": risk_analysis.confidence_score,
            "narrative": compliance_review.narrative,
            "key_indicators": risk_analysis.key_indicators,
            "ai_reasoning": risk_analysis.reasoning,
            "transaction_count": len(case_data.transactions),
            "absolute_transaction_total": sum(abs(txn.amount) for txn in case_data.transactions),
            "activity_start_date": min(txn.transaction_date for txn in case_data.transactions),
            "activity_end_date": max(txn.transaction_date for txn in case_data.transactions),
        },
        "regulatory_compliance": {
            "citations": compliance_review.regulatory_citations,
            "narrative_word_count": len(compliance_review.narrative.split()),
            "completeness_check": compliance_review.completeness_check,
            "compliance_status": "approved",
        },
        "filing_institution": {
            "name": os.getenv("FILING_INSTITUTION_NAME", "Demo Financial Institution"),
            "contact_role": "BSA/AML Compliance Officer",
        },
        "audit_trail": {
            "case_id": case_data.case_id,
            "processing_date": filing_date,
            "agent_decisions": {
                "risk_analyst": {
                    "classification": risk_analysis.classification,
                    "confidence_score": risk_analysis.confidence_score,
                    "risk_level": risk_analysis.risk_level,
                },
                "compliance_officer": {
                    "completeness_check": compliance_review.completeness_check,
                    "citations": compliance_review.regulatory_citations,
                },
            },
            "human_review": {
                "decision": human_review["decision"],
                "reviewer_id": human_review["reviewer_id"],
                "rationale": human_review["rationale"],
                "decision_timestamp": human_review["decision_timestamp"],
            },
        },
    }
    canonical = json.dumps(document, sort_keys=True, separators=(",", ":"), default=str)
    document["sar_metadata"]["sha256"] = hashlib.sha256(canonical.encode()).hexdigest()
    return document


def save_sar_document(sar_document, output_dir=OUTPUT_DIR / "filed_sars"):
    """Validate and atomically save a SAR document; return its path."""
    sar_id = sar_document.get("sar_metadata", {}).get("sar_id")
    if not sar_id:
        raise ValueError("SAR document is missing sar_metadata.sar_id")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    destination = output_dir / f"{sar_id}.json"
    temporary = destination.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(sar_document, indent=2, default=str), encoding="utf-8")
    temporary.replace(destination)
    return destination


print("📄 SAR document generation functions defined")

## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [ ]:
def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions):
    """Calculate processing, decision, compliance, and avoided-call metrics."""
    total_cases = len(processed_cases)
    approved_count = len(approved_sars)
    rejected_count = len(rejected_cases)
    completed_decisions = [d for d in audit_decisions if d.get("decision") != "ERROR"]
    errors = [d for d in audit_decisions if d.get("decision") == "ERROR"]
    timings = [float(d.get("processing_time_ms", 0)) for d in audit_decisions]
    approval_rate = approved_count / total_cases if total_cases else 0.0
    rejection_rate = rejected_count / total_cases if total_cases else 0.0
    metrics = {
        "total_cases": total_cases,
        "approved_sars": approved_count,
        "rejected_cases": rejected_count,
        "error_count": len(errors),
        "approval_rate": approval_rate,
        "rejection_rate": rejection_rate,
        "average_processing_time_ms": sum(timings) / len(timings) if timings else 0.0,
        "compliance_calls_executed": approved_count,
        "compliance_calls_avoided": rejected_count,
        "compliance_call_savings_rate": rejection_rate,
        "decision_count": len(completed_decisions),
    }
    print(json.dumps(metrics, indent=2))
    return metrics


def validate_ai_decisions(audit_decisions):
    """Summarize AI classifications, confidence, and human rejection patterns."""
    valid = [d for d in audit_decisions if d.get("decision") in {"PROCEED", "REJECT"}]
    classifications = {}
    for decision in valid:
        label = decision.get("ai_classification", "Unknown")
        classifications[label] = classifications.get(label, 0) + 1
    confidences = [float(d["ai_confidence"]) for d in valid if d.get("ai_confidence") is not None]
    rejections = sum(d["decision"] == "REJECT" for d in valid)
    result = {
        "classification_distribution": classifications,
        "average_confidence": sum(confidences) / len(confidences) if confidences else None,
        "minimum_confidence": min(confidences) if confidences else None,
        "maximum_confidence": max(confidences) if confidences else None,
        "human_rejection_rate": rejections / len(valid) if valid else 0.0,
    }
    print(json.dumps(result, indent=2))
    return result


def save_workflow_report(metrics, decision_analysis, audit_decisions,
                         output_dir=OUTPUT_DIR / "audit_reports"):
    """Persist an examination-ready workflow summary using an atomic write."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    created_at = datetime.now(timezone.utc).isoformat()
    report = {
        "report_id": f"WORKFLOW_{uuid.uuid4()}",
        "created_at": created_at,
        "metrics": metrics,
        "decision_analysis": decision_analysis,
        "case_decisions": audit_decisions,
    }
    filename = f"workflow_report_{created_at.replace(':', '').replace('+0000', 'Z')}.json"
    destination = output_dir / filename
    temporary = destination.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
    temporary.replace(destination)
    return destination


workflow_metrics = analyze_workflow_efficiency(
    processed_cases, approved_sars, rejected_cases, audit_decisions
)
decision_analysis = validate_ai_decisions(audit_decisions)

## 🏁 Step 6: Complete System Demonstration

Test your complete system with comprehensive scenarios to validate production readiness.

In [ ]:
def demonstrate_complete_system(decision_provider=None, top_n=3, reviewer_id="demo_reviewer"):
    """Run the complete workflow and persist its metrics and decision report."""
    customers, accounts, transactions = load_and_preprocess_data()
    candidates = screen_high_risk_customers(customers, accounts, transactions, top_n=top_n)
    results = run_two_stage_sar_workflow(
        candidates, decision_provider=decision_provider, reviewer_id=reviewer_id
    )
    processed, filed, rejected, decisions = results
    metrics = analyze_workflow_efficiency(processed, filed, rejected, decisions)
    validation = validate_ai_decisions(decisions)
    report_path = save_workflow_report(metrics, validation, decisions)
    return {
        "processed_cases": processed,
        "approved_sars": filed,
        "rejected_cases": rejected,
        "audit_decisions": decisions,
        "metrics": metrics,
        "decision_analysis": validation,
        "workflow_report_path": report_path,
    }


def alternating_demo_decision_provider():
    """Return a provider that demonstrates both approval and rejection paths."""
    state = {"count": 0}
    def decide(case_data, risk_analysis):
        state["count"] += 1
        approved = state["count"] % 2 == 1
        return {
            "decision": "yes" if approved else "no",
            "reviewer_id": "automated_demo_reviewer",
            "rationale": (
                "Approved for narrative generation based on the documented risk indicators."
                if approved else
                "Rejected during demonstration to verify the human gate and avoided compliance call."
            ),
        }
    return decide


print("🏁 Demonstration ready.")
print("Interactive: demonstrate_complete_system()")
print("Mixed decisions: demonstrate_complete_system(alternating_demo_decision_provider(), top_n=3)")

## Implementation Notes and Customizations

This implementation extends the starter workflow with:

- **Transparent screening:** deterministic risk scores combine customer rating, transaction volume, frequency, and recency.
- **Human decision records:** approvals and rejections capture reviewer ID, rationale, and an ISO-8601 timestamp.
- **Cost control:** the Compliance Officer agent runs only after approval; reports record executed and avoided calls.
- **Failure auditing:** orchestration errors are written to JSONL with the failed stage and error message.
- **Output integrity:** SAR files are atomically written and include a SHA-256 checksum plus agent and human decisions.
- **Examination artifacts:** each demonstration saves metrics and case decisions under `outputs/audit_reports/`.
- **Safe execution:** live API calls remain opt-in; automated tests can inject deterministic decision providers.

Use `alternating_demo_decision_provider()` to exercise both the approval and rejection branches without interactive input.

## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [ ] **Data Loading**: Load and preprocess CSV data with proper error handling
- [ ] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [ ] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [ ] **Human Decision Gates**: Implement interactive approval/rejection points
- [ ] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [ ] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [ ] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [ ] **Component Validation**: Verify all foundation components and agents are available
- [ ] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [ ] **End-to-End Testing**: Test complete workflow with automated scenarios
- [ ] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [ ] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [ ] **Error Handling**: Robust exception handling for all workflow steps
- [ ] **Data Validation**: Proper validation of all inputs and outputs
- [ ] **File Management**: Organize outputs in appropriate directories
- [ ] **Logging**: Comprehensive audit logging for compliance
- [ ] **Performance**: Efficient processing of multiple cases
- [ ] **User Experience**: Clear prompts and feedback for human reviewers
- [ ] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [ ] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [ ] **Cost Optimization**: Demonstrate savings from two-stage processing
- [ ] **Audit Readiness**: Create examination-ready documentation
- [ ] **Quality Assurance**: Validate AI decisions with human oversight
- [ ] **Scalability**: Design for processing larger datasets
- [ ] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [ ]:
# Workflow Integration Testing
import pytest

tests_path = PROJECT_ROOT / "tests"


def run_integration_tests():
    """Run all component tests once and return whether they pass."""
    result = pytest.main([str(tests_path), "-q", "--tb=short"])
    passed = result == pytest.ExitCode.OK
    print(f"{'✅' if passed else '❌'} Component test suite {'passed' if passed else 'failed'}")
    return passed


def validate_workflow_components():
    """Validate imports, data files, and agent initialization state."""
    status = {
        "foundation_sar": all((CustomerData, CaseData, ExplainabilityLogger, DataLoader)),
        "risk_analyst_agent": RiskAnalystAgent is not None,
        "compliance_officer_agent": ComplianceOfficerAgent is not None,
        "test_directory": tests_path.is_dir(),
        "data_files": all((DATA_DIR / name).is_file() for name in (
            "customers.csv", "accounts.csv", "transactions.csv"
        )),
        "openai_client": client is not None,
    }
    for component, ready in status.items():
        print(f"{'✅' if ready else '⚠️'} {component}: {ready}")
    # API credentials are needed for live E2E processing, but not component validation.
    return all(value for key, value in status.items() if key != "openai_client")


components_ready = validate_workflow_components()
# Run explicitly when desired: integration_tests_passed = run_integration_tests()

In [ ]:
def test_complete_workflow(decision_provider=None, top_n=2):
    """Execute an opt-in live end-to-end smoke test and validate generated artifacts."""
    if risk_agent is None or compliance_agent is None:
        print("⚠️ Skipped: OPENAI_API_KEY is required for the live end-to-end test")
        return False
    if decision_provider is None:
        decision_provider = lambda case, risk: "yes"

    result = demonstrate_complete_system(decision_provider, top_n=top_n)
    assert result["processed_cases"], "No cases were processed"
    assert len(result["audit_decisions"]) == len(result["processed_cases"])
    for sar in result["approved_sars"]:
        assert sar["sar_metadata"]["sha256"]
        assert sar["regulatory_compliance"]["narrative_word_count"] <= 120
        assert sar["regulatory_compliance"]["completeness_check"]
    print("✅ End-to-end workflow test passed")
    return True


# This makes external test discovery ignore the live helper.
test_complete_workflow.__test__ = False
print("🎯 Live end-to-end test ready; call test_complete_workflow() explicitly.")